In [1]:
import glob
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, LSTM

### Настройки

In [2]:
DATASET_DIR = "dataset"
ALL_DATASET_FILE = "all_data.csv"
EPOCHES_COUNT = 5
BATCH_SIZE = 256

all_dataset_path = os.path.join(DATASET_DIR, ALL_DATASET_FILE)
scaler = StandardScaler()

### Предобработка и создание единого датасета

In [3]:
files = [file for file in glob.glob(DATASET_DIR + "**/*.csv", recursive=False)]
files

['dataset\\all_data.csv',
 'dataset\\Friday-02-03-2018_TrafficForML_CICFlowMeter.csv',
 'dataset\\Friday-16-02-2018_TrafficForML_CICFlowMeter.csv',
 'dataset\\Friday-23-02-2018_TrafficForML_CICFlowMeter.csv',
 'dataset\\Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv',
 'dataset\\Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv',
 'dataset\\Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv',
 'dataset\\Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv',
 'dataset\\Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv',
 'dataset\\Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv']

In [4]:
if os.path.exists(all_dataset_path):
    df = pd.read_csv('dataset/all_data.csv') # pd.read_csv('dataset/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv') 
else:
    dframes = []
    for file in files:
        loaded = pd.read_csv(file)    
        dframes.append(loaded)
        print(file, 'загружен')

    df = pd.concat(dframes)
    df.to_csv('dataset/all_data.csv')

df

C:\Users\pc\AppData\Local\Temp\ipykernel_17200\1646178193.py:2: DtypeWarning: Columns (1,2,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('dataset/all_data.csv') # pd.read_csv('dataset/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv')


,Unnamed: 0,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0,443,6,02/03/2018 08:47:38,141385,9,7,553,3773.0,202,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,1,49684,6,02/03/2018 08:47:38,281,2,1,38,0.0,38,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,2,443,6,02/03/2018 08:47:40,279824,11,15,1086,10527.0,385,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,3,443,6,02/03/2018 08:47:40,132,2,0,0,0.0,0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,4,443,6,02/03/2018 08:47:41,274016,9,13,1285,6141.0,517,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8284249,613099,23,6,28/02/2018 11:59:12,3,1,1,0,0,0,...,24,0,0,0,0,0,0.0,0,0,Infilteration
8284250,613100,425,6,28/02/2018 10:50:04,2,1,1,0,0,0,...,24,0,0,0,0,0,0.0,0,0,Infilteration
8284251,613101,445,6,28/02/2018 12:52:55,732728,2,2,0,0,0,...,32,0,0,0,0,0,0.0,0,0,Benign
8284252,613102,23,6,28/02/2018 11:10:50,22,1,1,0,0,0,...,24,0,0,0,0,0,0.0,0,0,Infilteration


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8284254 entries, 0 to 8284253
Data columns (total 81 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   Unnamed: 0         int64 
 1   Dst Port           object
 2   Protocol           object
 3   Timestamp          object
 4   Flow Duration      object
 5   Tot Fwd Pkts       object
 6   Tot Bwd Pkts       object
 7   TotLen Fwd Pkts    object
 8   TotLen Bwd Pkts    object
 9   Fwd Pkt Len Max    object
 10  Fwd Pkt Len Min    object
 11  Fwd Pkt Len Mean   object
 12  Fwd Pkt Len Std    object
 13  Bwd Pkt Len Max    object
 14  Bwd Pkt Len Min    object
 15  Bwd Pkt Len Mean   object
 16  Bwd Pkt Len Std    object
 17  Flow Byts/s        object
 18  Flow Pkts/s        object
 19  Flow IAT Mean      object
 20  Flow IAT Std       object
 21  Flow IAT Max       object
 22  Flow IAT Min       object
 23  Fwd IAT Tot        object
 24  Fwd IAT Mean       object
 25  Fwd IAT Std        object
 26  Fwd IAT Max   

In [6]:
labels = list(df["Label"].unique())
labels

['Benign',
 'Bot',
 'DoS attacks-SlowHTTPTest',
 'DoS attacks-Hulk',
 'Label',
 'Brute Force -Web',
 'Brute Force -XSS',
 'SQL Injection',
 'Infilteration',
 'DoS attacks-GoldenEye',
 'DoS attacks-Slowloris',
 'FTP-BruteForce',
 'SSH-Bruteforce',
 'DDOS attack-LOIC-UDP',
 'DDOS attack-HOIC']

### Чистка

In [7]:
df.drop('Timestamp', axis=1, inplace=True)

# df['Timestamp'] = df['Timestamp'].astype(str)
# df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M:%S').astype(int)


In [8]:
df.drop('Unnamed: 0', axis=1, inplace=True)

In [9]:
df.drop(['Flow ID', 'Src IP', 'Dst IP'], axis=1, inplace=True)

KeyError: "['Flow ID', 'Src IP', 'Dst IP'] not found in axis"

In [10]:
df

,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443,6,141385,9,7,553,3773.0,202,0,61.444444,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,49684,6,281,2,1,38,0.0,38,0,19.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,443,6,279824,11,15,1086,10527.0,385,0,98.727273,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,443,6,132,2,0,0,0.0,0,0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,443,6,274016,9,13,1285,6141.0,517,0,142.777778,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8284249,23,6,3,1,1,0,0,0,0,0,...,24,0,0,0,0,0,0.0,0,0,Infilteration
8284250,425,6,2,1,1,0,0,0,0,0,...,24,0,0,0,0,0,0.0,0,0,Infilteration
8284251,445,6,732728,2,2,0,0,0,0,0,...,32,0,0,0,0,0,0.0,0,0,Benign
8284252,23,6,22,1,1,0,0,0,0,0,...,24,0,0,0,0,0,0.0,0,0,Infilteration


In [11]:
df['Label'] = df['Label'].astype('category')
df[['Label']] = df[['Label']].apply(lambda col: pd.Categorical(col).codes)
df

,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443,6,141385,9,7,553,3773.0,202,0,61.444444,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,49684,6,281,2,1,38,0.0,38,0,19.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,443,6,279824,11,15,1086,10527.0,385,0,98.727273,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,443,6,132,2,0,0,0.0,0,0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,443,6,274016,9,13,1285,6141.0,517,0,142.777778,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8284249,23,6,3,1,1,0,0,0,0,0,...,24,0,0,0,0,0,0.0,0,0,11
8284250,425,6,2,1,1,0,0,0,0,0,...,24,0,0,0,0,0,0.0,0,0,11
8284251,445,6,732728,2,2,0,0,0,0,0,...,32,0,0,0,0,0,0.0,0,0,0
8284252,23,6,22,1,1,0,0,0,0,0,...,24,0,0,0,0,0,0.0,0,0,11


In [12]:
labels_nums = list(df["Label"].unique())
labels_nums

[0, 1, 8, 7, 12, 2, 3, 13, 11, 6, 9, 10, 14, 5, 4]

In [13]:
df = df[df['Label'] != 12]

In [14]:
for col in df.columns:
    df[col] = df[col].astype(float)

C:\Users\pc\AppData\Local\Temp\ipykernel_17200\2631235247.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype(float)
C:\Users\pc\AppData\Local\Temp\ipykernel_17200\2631235247.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype(float)
C:\Users\pc\AppData\Local\Temp\ipykernel_17200\2631235247.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats

In [15]:
df

,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443.0,6.0,141385.0,9.0,7.0,553.0,3773.0,202.0,0.0,61.444444,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,49684.0,6.0,281.0,2.0,1.0,38.0,0.0,38.0,0.0,19.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,443.0,6.0,279824.0,11.0,15.0,1086.0,10527.0,385.0,0.0,98.727273,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,443.0,6.0,132.0,2.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,443.0,6.0,274016.0,9.0,13.0,1285.0,6141.0,517.0,0.0,142.777778,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8284249,23.0,6.0,3.0,1.0,1.0,0.0,0.0,0.0,0.0,0.000000,...,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0
8284250,425.0,6.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.000000,...,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0
8284251,445.0,6.0,732728.0,2.0,2.0,0.0,0.0,0.0,0.0,0.000000,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8284252,23.0,6.0,22.0,1.0,1.0,0.0,0.0,0.0,0.0,0.000000,...,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0


In [16]:
df.fillna(df.mean(), inplace=True)
df

C:\Users\pc\AppData\Local\Temp\ipykernel_17200\3921218131.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.fillna(df.mean(), inplace=True)


,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443.0,6.0,141385.0,9.0,7.0,553.0,3773.0,202.0,0.0,61.444444,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,49684.0,6.0,281.0,2.0,1.0,38.0,0.0,38.0,0.0,19.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,443.0,6.0,279824.0,11.0,15.0,1086.0,10527.0,385.0,0.0,98.727273,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,443.0,6.0,132.0,2.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,443.0,6.0,274016.0,9.0,13.0,1285.0,6141.0,517.0,0.0,142.777778,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8284249,23.0,6.0,3.0,1.0,1.0,0.0,0.0,0.0,0.0,0.000000,...,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0
8284250,425.0,6.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.000000,...,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0
8284251,445.0,6.0,732728.0,2.0,2.0,0.0,0.0,0.0,0.0,0.000000,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8284252,23.0,6.0,22.0,1.0,1.0,0.0,0.0,0.0,0.0,0.000000,...,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0


In [17]:
df.drop_duplicates(inplace=True)
df

C:\Users\pc\AppData\Local\Temp\ipykernel_17200\1725291922.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443.0,6.0,141385.0,9.0,7.0,553.0,3773.0,202.0,0.0,61.444444,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,49684.0,6.0,281.0,2.0,1.0,38.0,0.0,38.0,0.0,19.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,443.0,6.0,279824.0,11.0,15.0,1086.0,10527.0,385.0,0.0,98.727273,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,443.0,6.0,132.0,2.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,443.0,6.0,274016.0,9.0,13.0,1285.0,6141.0,517.0,0.0,142.777778,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8284229,7555.0,6.0,105445.0,2.0,1.0,0.0,0.0,0.0,0.0,0.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8284232,445.0,6.0,733880.0,2.0,2.0,0.0,0.0,0.0,0.0,0.000000,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8284241,1033.0,6.0,7.0,1.0,1.0,0.0,0.0,0.0,0.0,0.000000,...,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0
8284251,445.0,6.0,732728.0,2.0,2.0,0.0,0.0,0.0,0.0,0.000000,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5883327 entries, 0 to 8284252
Data columns (total 79 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Dst Port           float64
 1   Protocol           float64
 2   Flow Duration      float64
 3   Tot Fwd Pkts       float64
 4   Tot Bwd Pkts       float64
 5   TotLen Fwd Pkts    float64
 6   TotLen Bwd Pkts    float64
 7   Fwd Pkt Len Max    float64
 8   Fwd Pkt Len Min    float64
 9   Fwd Pkt Len Mean   float64
 10  Fwd Pkt Len Std    float64
 11  Bwd Pkt Len Max    float64
 12  Bwd Pkt Len Min    float64
 13  Bwd Pkt Len Mean   float64
 14  Bwd Pkt Len Std    float64
 15  Flow Byts/s        float64
 16  Flow Pkts/s        float64
 17  Flow IAT Mean      float64
 18  Flow IAT Std       float64
 19  Flow IAT Max       float64
 20  Flow IAT Min       float64
 21  Fwd IAT Tot        float64
 22  Fwd IAT Mean       float64
 23  Fwd IAT Std        float64
 24  Fwd IAT Max        float64
 25  Fwd IAT Min        floa

In [19]:
df = df.dropna()
df

,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443.0,6.0,141385.0,9.0,7.0,553.0,3773.0,202.0,0.0,61.444444,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,49684.0,6.0,281.0,2.0,1.0,38.0,0.0,38.0,0.0,19.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,443.0,6.0,279824.0,11.0,15.0,1086.0,10527.0,385.0,0.0,98.727273,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,443.0,6.0,132.0,2.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,443.0,6.0,274016.0,9.0,13.0,1285.0,6141.0,517.0,0.0,142.777778,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8284229,7555.0,6.0,105445.0,2.0,1.0,0.0,0.0,0.0,0.0,0.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8284232,445.0,6.0,733880.0,2.0,2.0,0.0,0.0,0.0,0.0,0.000000,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8284241,1033.0,6.0,7.0,1.0,1.0,0.0,0.0,0.0,0.0,0.000000,...,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0
8284251,445.0,6.0,732728.0,2.0,2.0,0.0,0.0,0.0,0.0,0.000000,...,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
df.replace([np.inf, -np.inf], 0, inplace=True)

In [21]:
df_normalized = pd.DataFrame(scaler.fit_transform(df)) 
df_normalized

,0,1,2,3,4,5,6,7,8,9,...,69,70,71,72,73,74,75,76,77,78
0,-0.639352,-0.425249,-0.017200,-0.015619,-0.005739,-0.012589,-0.008779,-0.318409,-0.315756,-0.146106,...,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949,-0.322391
1,1.540222,-0.425249,-0.017373,-0.018998,-0.035566,-0.018169,-0.022008,-0.767796,-0.315756,-0.725467,...,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949,-0.322391
2,-0.639352,-0.425249,-0.017032,-0.014654,0.034031,-0.006815,0.014902,0.183041,-0.315756,0.362799,...,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949,-0.322391
3,-0.639352,-0.425249,-0.017373,-0.018998,-0.040537,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949,-0.322391
4,-0.639352,-0.425249,-0.017039,-0.015619,0.024088,-0.004659,-0.000476,0.544743,-0.315756,0.964082,...,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949,-0.322391
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5883322,-0.324551,-0.425249,-0.017244,-0.018998,-0.035566,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949,-0.322391
5883323,-0.639264,-0.425249,-0.016478,-0.018998,-0.030595,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,1.800141,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949,-0.322391
5883324,-0.613237,-0.425249,-0.017373,-0.019481,-0.035566,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,0.671472,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949,3.713302
5883325,-0.639264,-0.425249,-0.016479,-0.018998,-0.030595,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,1.800141,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949,-0.322391


Обучение

In [22]:
X_train, X_test, y_train, y_test = train_test_split(df_normalized.drop([78], axis=1), df['Label'], test_size=0.3, random_state=42)
X_train

,0,1,2,3,4,5,6,7,8,9,...,68,69,70,71,72,73,74,75,76,77
3412812,1.602943,-0.425249,-0.017373,-0.018998,-0.040537,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,-0.017879,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
4750077,-0.656615,2.149081,-0.017357,-0.019481,-0.035566,-0.018223,-0.021587,-0.781497,1.024872,-0.534369,...,-0.017879,-1.585865,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
4470964,-0.656615,2.149081,-0.017141,-0.018998,-0.030595,-0.017866,-0.021229,-0.781497,1.024872,-0.534369,...,-0.017396,-1.585865,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
2816411,1.661769,-0.425249,-0.017373,-0.019481,-0.035566,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,-0.017879,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
2504527,-0.639352,-0.425249,-0.010807,-0.015619,0.009175,-0.001571,-0.002464,0.911925,-0.315756,1.396328,...,-0.015947,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1570006,-0.639352,-0.425249,0.125415,-0.009827,0.078772,0.015179,-0.007787,2.950609,-0.315756,1.040565,...,-0.010634,0.107138,-0.038735,0.075604,0.120163,-0.051910,0.006387,0.002616,0.002158,-0.005694
2234489,1.669825,-0.425249,-0.017372,-0.018516,-0.040537,-0.018245,-0.022008,-0.786978,-0.315756,-0.843765,...,-0.017396,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
4926484,-0.655420,-0.425249,-0.017331,-0.018516,-0.020652,-0.014908,-0.018730,0.056994,-0.315756,0.557620,...,-0.017396,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
4304572,-0.657987,-0.425249,-0.016864,-0.009344,0.058887,0.002481,-0.012664,0.881784,-0.315756,0.221337,...,-0.010150,1.800141,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949


In [23]:
X_test

,0,1,2,3,4,5,6,7,8,9,...,68,69,70,71,72,73,74,75,76,77
764534,-0.508952,-0.425249,-0.012389,-0.013206,0.004204,-0.002763,-0.015939,1.158540,-0.315756,0.438671,...,-0.014498,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
3206194,-0.454995,-0.425249,-0.017373,-0.018998,-0.040537,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,-0.017879,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
5752741,-0.639352,-0.425249,0.056740,-0.011758,0.073800,-0.002329,0.040319,1.059894,-0.315756,0.219585,...,-0.013532,0.107138,-0.041106,0.045524,0.029487,-0.051718,0.007485,-0.001457,0.002346,0.035350
2054927,1.698552,-0.425249,-0.017373,-0.019481,-0.035566,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,-0.017879,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
3380611,-0.508952,-0.425249,-0.014648,-0.016102,-0.005739,-0.006316,-0.016465,0.939327,-0.315756,0.946641,...,-0.015464,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434676,0.288453,-0.425249,0.087776,-0.018998,-0.040537,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,-0.017879,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,0.181623,-0.001948,0.075520,0.768399
3954666,-0.092168,-0.425249,0.088118,-0.018998,-0.040537,-0.018581,-0.022008,-0.871923,-0.315756,-0.984814,...,-0.017879,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,0.182238,-0.001948,0.075779,0.770976
3207570,-0.656615,2.149081,-0.017328,-0.019481,-0.035566,-0.018126,-0.021636,-0.756836,1.390498,-0.411520,...,-0.017879,-1.585865,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949
4992354,-0.655420,-0.425249,-0.017367,-0.018516,-0.020652,-0.014973,-0.018730,0.040553,-0.315756,0.530320,...,-0.017396,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949


In [24]:
y_train

4501313     0.0
6466081     0.0
6103709     0.0
3733859     0.0
3360856     0.0
           ... 
2263695     0.0
3038025     0.0
6703304     4.0
5821598    14.0
2402432     0.0
Name: Label, Length: 4118328, dtype: float64

In [25]:
y_test

983573     0.0
4223672    0.0
8112932    0.0
2822307    0.0
4459000    0.0
          ... 
601611     0.0
5179755    0.0
4225369    0.0
6816676    4.0
2445055    0.0
Name: Label, Length: 1764999, dtype: float64

In [26]:
x_size = X_train.shape[1]
x_size

78

In [27]:
y_size = len(labels)
y_size

15

In [28]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
y_train, y_test

(array([[1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 1.],
        [1., 0., 0., ..., 0., 0., 0.]], dtype=float32),
 array([[1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        ...,
        [1., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.]], dtype=float32))

In [29]:
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(x_size, 1)))
model.add(LSTM(units=50))
model.add(Dense(units=y_size, activation='softmax'))

# Компиляция модели
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Обучение модели
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHES_COUNT, batch_size=BATCH_SIZE)

# Оценка качества модели на тестовой выборке
score = model.evaluate(X_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])



Epoch 1/5


16088/16088 [==============================] - 2727s 169ms/step - loss: 0.2335 - accuracy: 0.9457 - val_loss: 0.1034 - val_accuracy: 0.9724
Epoch 2/5
16088/16088 [==============================] - 2900s 180ms/step - loss: 0.1004 - accuracy: 0.9729 - val_loss: 0.0991 - val_accuracy: 0.9732
Epoch 3/5
16088/16088 [==============================] - 2978s 185ms/step - loss: 0.0963 - accuracy: 0.9738 - val_loss: 0.0963 - val_accuracy: 0.9737
Epoch 4/5
16088/16088 [==============================] - 3023s 188ms/step - loss: 0.0944 - accuracy: 0.9742 - val_loss: 0.0942 - val_accuracy: 0.9743
Epoch 5/5
16088/16088 [==============================] - 3071s 191ms/step - loss: 0.0932 - accuracy: 0.9744 - val_loss: 0.0930 - val_accuracy: 0.9742
Test loss: 0.09300019592046738
Test accuracy: 0.9741756319999695


In [31]:
model.save('model.keras')

Тест

In [38]:
df2 = pd.read_csv('test.csv')
df2

,Unnamed: 0,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,tot_fwd_pkts,...,bwd_pkts_b_avg,fwd_blk_rate_avg,bwd_blk_rate_avg,fwd_seg_size_avg,bwd_seg_size_avg,cwe_flag_count,subflow_fwd_pkts,subflow_bwd_pkts,subflow_fwd_byts,subflow_bwd_byts
0,0,46159,443,6,1.135206e+04,2.933388e+04,440.448608,176.179443,264.269165,2,...,0.0,0.000000e+00,0.000000e+00,54.000000,75.000000,0,2,3,108,225
1,1,46160,443,6,1.956639e+05,8.279503e+02,15.332412,10.221608,5.110804,2,...,0.0,0.000000e+00,0.000000e+00,54.000000,54.000000,0,2,1,108,54
2,2,80,44783,6,3.947496e+04,2.363523e+04,151.995072,75.997536,75.997536,3,...,0.0,0.000000e+00,0.000000e+00,136.000000,175.000000,0,3,3,408,525
3,3,44839,445,6,1.427889e+03,8.474049e+04,1400.669227,700.334613,700.334613,1,...,0.0,0.000000e+00,0.000000e+00,55.000000,66.000000,0,1,1,55,66
4,4,46170,80,6,1.362731e+05,6.039341e+03,58.705624,36.691015,22.014609,5,...,0.0,0.000000e+00,0.000000e+00,129.800000,58.000000,0,5,3,649,174
5,5,46175,80,6,1.343560e+05,7.078209e+03,59.543293,37.214558,22.328735,5,...,0.0,0.000000e+00,0.000000e+00,155.400000,58.000000,0,5,3,777,174
6,6,46176,80,6,1.332400e+05,6.176824e+03,60.042036,37.526273,22.515764,5,...,0.0,0.000000e+00,0.000000e+00,129.800000,58.000000,0,5,3,649,174
7,7,46177,80,6,1.341262e+05,6.434239e+03,59.645325,37.278328,22.366997,5,...,0.0,0.000000e+00,0.000000e+00,137.800000,58.000000,0,5,3,689,174
8,8,46178,80,6,1.346600e+05,6.379028e+03,59.408879,37.130549,22.278330,5,...,0.0,0.000000e+00,0.000000e+00,137.000000,58.000000,0,5,3,685,174
9,9,61449,1947,17,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,1,...,0.0,0.000000e+00,0.000000e+00,82.000000,0.000000,0,1,0,82,0


In [39]:
df2.drop(['src_port', 'Unnamed: 0'], axis=1, inplace=True)
df2 = pd.DataFrame(scaler.fit_transform(df)[0])
df2

,0
0,-0.639352
1,-0.425249
2,-0.017200
3,-0.015619
4,-0.005739
...,...
74,-0.015627
75,-0.001948
76,-0.007475
77,-0.058949


In [42]:
df2_transposed = df2.T
df2_transposed

,0,1,2,3,4,5,6,7,8,9,...,69,70,71,72,73,74,75,76,77,78
0,-0.639352,-0.425249,-0.0172,-0.015619,-0.005739,-0.012589,-0.008779,-0.318409,-0.315756,-0.146106,...,0.107138,-0.072274,-0.058005,-0.086453,-0.059491,-0.015627,-0.001948,-0.007475,-0.058949,-0.322391


In [45]:
predicted = model.predict(df2_transposed)
predicted

1/1 [==============================] - 0s 60ms/step


array([[8.6875051e-01, 6.2785080e-06, 3.5581569e-04, 3.8350354e-05,
        1.6190814e-12, 9.1003145e-08, 4.4768373e-11, 4.9575486e-13,
        6.3603001e-10, 1.0901665e-06, 3.7556308e-10, 1.3062599e-01,
        1.4957459e-10, 2.2182288e-04, 4.8802123e-10]], dtype=float32)

In [48]:
df3 = pd.DataFrame(predicted).T
df3

,0
0,8.687505e-01
1,6.278508e-06
2,3.558157e-04
3,3.835035e-05
4,1.619081e-12
5,9.100314e-08
6,4.476837e-11
7,4.957549e-13
8,6.360300e-10
9,1.090167e-06


In [50]:
labels[max(df3)]

'Benign'